In [1]:
import matplotlib.pyplot as plt
from astropy.cosmology import FlatLambdaCDM
from astropy.units import Quantity
from slsim.lens_pop import LensPop
import numpy as np
import corner
import slsim.Pipelines as pipelines
import slsim.Sources as sources
import slsim.Deflectors as deflectors

ModuleNotFoundError: No module named 'slsim'

# Galaxy-galaxy simulations

This notebook walks through the basics of simulating a galaxy-galaxy strong lensing population.
The underlying 

assumptions of the galaxy populations (for both lenses and sources) are drawn from a population pre-configured

 and rendered through [SkyPy](https://github.com/skypyproject/skypy). The specific settings are described in the [readme file](https://github.com/LSST-strong-lensing/slsim/tree/gg-lensing/data/SkyPy).

The notebook goes in three steps:

1. The populations of lenses and sources is produced.
2. Random draws of the population are generated and realized as images
3. The full population is generated in catalogue form
4. the full population is represented in a corner plot



## Generate population of galaxies and (potential) deflectors
The LensPop() class in the slsim package is used to produce a set of galaxies (as lenses and sources)

as seen on the sky within a certain sky area.
We use the default SkyPy configuration file. Alternative configuration

files can be used.

In [ ]:
# define a cosmology
cosmo = FlatLambdaCDM(H0=70, Om0=0.3)

# define a sky area. One can define sky_area for galaxy simulation and lens population separately. 
# One can give the same value or different value.
sky_area = Quantity(value=2, unit="deg2")
# One can choose required sky area.
full_sky_area = Quantity(value=50, unit="deg2")

# define limits in the intrinsic deflector and source population (in addition to the skypy config
# file). One can apply their own limits here.
kwargs_deflector_cut = {"band": "g", "band_max": 28, "z_min": 0.01, "z_max": 1.5}
kwargs_source_cut = {"band": "g", "band_max": 28, "z_min": 0.01, "z_max": 1.5}

In [ ]:
# Generate galaxy population using skypy pipeline.
galaxy_simulation_pipeline = pipelines.SkyPyPipeline(
    skypy_config=None, sky_area=sky_area, filters=None, cosmo=cosmo
)

In [ ]:
# Initiate deflector population class.
gamma_dist={"mean": 2.10, "std_dev": 0.16}
lens_galaxies = deflectors.AllLensGalaxies(
    red_galaxy_list=galaxy_simulation_pipeline.red_galaxies,
    blue_galaxy_list=galaxy_simulation_pipeline.blue_galaxies,
    kwargs_cut=kwargs_deflector_cut,
    kwargs_mass2light=None,
    cosmo=cosmo,
    sky_area=sky_area,
    gamma_pl=gamma_dist
)

In [ ]:
# Initiate source population class.
source_galaxies = sources.Galaxies(
    galaxy_list=galaxy_simulation_pipeline.blue_galaxies,
    kwargs_cut=kwargs_source_cut,
    cosmo=cosmo,
    sky_area=sky_area,
    catalog_type="skypy",
    downsample_to_dc2=False
)

In [ ]:
# make galaxy-galaxy population class using LensPop
gg_lens_pop = LensPop(
    deflector_population=lens_galaxies,
    source_population=source_galaxies,
    cosmo=cosmo,
    sky_area=full_sky_area,
)

## Draw lens sample

In [ ]:
kwargs_lens_cut = {"min_image_separation": 1, "max_image_separation":8,"mag_arc_limit": {"g": 27}}
lens_population = gg_lens_pop.draw_population(kwargs_lens_cuts=kwargs_lens_cut, multi_source=True)

In [ ]:
## extract double source lenses
double_source_lens=[]
for i in range(len(lens_population)):
    if lens_population[i].source_number ==2:
        double_source_lens.append(lens_population[i])

In [ ]:
vel_disp=[]
m_star=[]
theta_e=[]
zl=[]
zs=[]
source_mag=[]
deflector_mag=[]
for gg_lens in double_source_lens:
    vel_disp.append(gg_lens.deflector_velocity_dispersion())
    m_star.append(np.log10(gg_lens.deflector_stellar_mass()))
    theta_e.append(gg_lens.einstein_radius)
    zl.append(gg_lens.deflector_redshift)
    zs.append(gg_lens.source_redshift_list)
    source_mag.append(gg_lens.extended_source_magnitude(band="g", lensed=True))
    deflector_mag.append(gg_lens.deflector_magnitude(band="g"))

In [ ]:
from astropy.table import Table

In [ ]:
data={"z_l": zl, "z_s": zs, "vel_disp":vel_disp, "logM": m_star, "theta_e": theta_e, "source_mag":source_mag, "lens_mag": deflector_mag}

In [ ]:
data_table=Table(data)

In [ ]:
data_table